# The distributional (LSS) engine

`gamboostLSS`/`glmboostLSS` use a multi-predictor engine layered on the same base
learners. The core abstractions are a weight-independent `LearnerBlueprint`,
the `LssFamily` trait, and two driver loops: `fit_lss` (cyclic) and
`fit_lss_noncyclic`.

## Learner blueprints

A distributional fit may use different learner sets for different parameters,
and a single parameter formula may mix `Bols`, `Bbs`, `Bns`, `Brandom`, `Bmrf`, `Bspatial`, `Bmono`, `Bconstrained`, and `Btree`. The frontend converts
every term to a `LearnerBlueprint` before fitting. A blueprint owns information
that does not depend on observation weights: the raw covariate for `Bols`, or
the B-spline basis plus the difference penalty for `Bbs`, the B-spline basis
and identity penalty for `Bns`, or the group indicator layout and identity
penalty for `Brandom`, the indicator layout, graph penalty, and optional spectral transform for `Bmrf`, the two marginal bases, tensor design, additive penalty, and optional spectral transform for `Bspatial`, the spline and constraint options for `Bmono` and `Bconstrained`, or the design and split controls for `Btree`. Calling `prepare(weights)`
constructs the weight-dependent centering, smoothing parameter, Gram matrix,
factorization, and work buffers.

This split matters during resampling. Reusing a full-sample prepared spline in a
fold would retain the wrong centering and the wrong value of lambda implied by
`df`. Rebuilding the entire spline basis in every fold would be correct but
wasteful. The blueprint caches the basis and penalty, then prepares only the
weight-dependent numerical state for each fold. It also returns a basis recipe
used by training replay, path truncation, and new-data prediction. Consequently,
a centered spline is fitted and predicted through the same spectral transform.
For `Brandom`, the recipe retains the fitted integer codes and raises on
an unknown group rather than silently assigning an arbitrary effect. `Bmrf`
replay applies the same graph-eigenvector transform used during fitting and
rejects region codes that do not index the supplied neighborhood matrix. `Bspatial` replay rebuilds both marginal bases with the training boundaries, forms their row-wise tensor products, and applies the same spectral map as the fit. `Bmono` and `Bconstrained` rebuild the raw spline basis for replay; the constraint affects the fitted coefficient increments, while each fold rebuilds the constrained quadratic solver from its in-bag weights. `Bconstrained` additionally validates that the caller's matrix has one column for every raw spline coefficient. `Btree` decodes each selected step's node vector and evaluates it on the packed row-major new-data design; node vectors remain separate because successive trees need not have the same topology.

## `LssFamily`

A distributional family declares $K$ parameters, each with a link, an offset rule,
a **partial** negative gradient, an inverse link (`response`), and a joint risk.
The offset function receives all parameter predictors:

```text
fn offset(&self, k, y, w, prior) -> f64
```

`prior` holds the already-computed link-scale offsets of parameters $0..k$. This
encodes the **sequential** offset computation: parameter $k$'s offset uses the
offsets just computed for earlier parameters and family defaults for later ones —
exactly R's shared-environment (`<<-`) behavior. Incorrect `prior` threading produces visibly off paths; e.g. `StudentTLSS`'s
$\mu$ offset is a robust location, not the mean, so a naive scheme fails.

## Cyclic driver

`fit_lss` keeps a predictor `eta[k]` per parameter, initialized to its offset. Per
iteration it loops over parameters in order; for each it computes that parameter's
partial gradient **using the current `eta` of all parameters** (so later
parameters in the same cycle see earlier ones' updates), fits its learners,
selects the best, and updates `eta[k]`. Per-parameter step budgets are supported:
parameter $k$ stops updating once its own `mstop[k]` is reached, while others
continue. A requested budget vector therefore defines its own interleaved path:
stopping one parameter changes the gradients seen by later updates of the other
parameters. This path dependence is central to cross-validation.

## Non-cyclic driver

`fit_lss_noncyclic` {cite}`thomas2018gradient` computes the best candidate update
for **every** parameter at the current state, then applies only the one whose
update most reduces the **joint** risk. The per-parameter selection counts sum to
`mstop`, so one scalar tunes all parameters. The implementation evaluates each
candidate's risk delta by temporarily applying it to a cloned `eta` slice.

## SSE-by-identity selection

The inner selection loop avoids $O(n)$ per-learner residual projection. At
iteration $m$ for parameter $k$, the gradient vector $u_k$ is computed once
($O(n)$). Each learner solves its penalized normal equation

$$G_j \beta_j = X_j^\top W u_k$$

and returns a candidate $\beta_j$ and the SSE:

$$\text{SSE}_j = \sum_i w_i u_{k,i}^2 - \beta_j^\top G_j \beta_j$$

The second term is $O(p_j^2)$ via the already-factorized $G_j$ — no per-observation
work. The winner is $\arg\min_j \text{SSE}_j$ (tie-break by lowest index). Only the
winner's $n$-length prediction $X_j \beta_j$ is formed.

```rust
// lss.rs:194-209 — SSE-by-identity per-parameter selection
for (li, learner) in learners_per_param[p].iter_mut().enumerate() {
    let (beta, sse) = learner.fit_step_select(u_view, weights, wu2);
    if sse < best_sse {
        best_sse = sse;
        best_idx = li;
        best_beta = beta.to_vec();
    }
}
// Winner prediction (O(n) once):
learners_per_param[p][best_idx].predict_into(&best_beta, &mut best_pred);
```


## NaN fallback

If any learner's Gram matrix is singular (zero weight in a fold, or degenerate
gradient), `fit_step_select` returns `sse = NaN`. The per-parameter inner loop
short-circuits on the first NaN and falls back to the explicit
`fit_step_ref(u_view, weights)` path:

```rust
// lss.rs:212-233 — NaN fallback to explicit fit_step_ref
if sse_is_nan {
    for (li, learner) in learners_per_param[p].iter_mut().enumerate() {
        let (beta, pred, sse) = learner.fit_step_ref(u_view, weights);
        if sse < best_sse { ... }
    }
}
```


If even the fallback produces no finite SSE, the step is recorded as an explicit
no-op (zero coefficients, zero prediction) rather than silently reusing stale
buffers from the previous iteration.

## Cyclic driver algorithm

`fit_lss` (`lss.rs:112-292`) runs the round-robin loop:

```
1. Compute sequential offsets (see below); initialize eta[k] = offset[k]
2. risk[0] = family.risk(y, etas, w)
3. for m = 0..max(mstops):
     for p = 0..K-1:
       if m >= mstops[p]: continue   // per-parameter budget exhausted
       u = family.ngradient(p, y, etas, w)   // partial gradient at current etas
       wu2 = sum(w * u^2)                    // O(n) once per parameter
       best = argmin_j SSE_j via SSE-by-identity
       if all SSEs are NaN: fallback to fit_step_ref
       eta[p] += nu * X_best * beta_best     // in-place O(n)
       record selected[p], coefs[p]
     risk[m+1] = family.risk(y, etas, w)
```

Key properties:
- **ETA IS SHARED**: `ngradient(p)` sees the most recent `eta[0..K-1]`, not the
  state at cycle start. Parameters earlier in the same cycle have already had
  their update applied; later ones still show the previous cycle's value.
- **Per-parameter `mstop[k]`**: a parameter stops updating once its own budget is
  exhausted, while others continue. Diagonal budgets $(m,\ldots,m)$ share
  checkpoints of one path; unequal budgets generally require separate fits.
- **Per-step increments not stored**: `BoostingPath.increments` is left empty.
  Predictions reconstruct from `offset + nu * sum` of winner predictions, so
  storing them is overhead (see `LSS4` comment at `lss.rs:280`).

## Non-cyclic driver algorithm

`fit_lss_noncyclic` (`lss.rs:299-460`, {cite}`thomas2018gradient`) selects only
one parameter to update per global iteration:

```
1. Compute sequential offsets; initialize eta[k] = offset[k]
2. risk[0] = family.risk(y, etas, w)
3. for step = 0..mstop-1:
     for p = 0..K-1:
       u = family.ngradient(p, y, etas, w)
       wu2 = sum(w * u^2)
       cand[p] = argmin_j SSE_j via SSE-by-identity
       cand_pred[p] = X_best * beta_best
     // Trial-apply each candidate to a clone of etas
     for p = 0..K-1:
       eta' = eta;  eta'[p] += nu * cand_pred[p]
       r_p = family.risk(y, eta', w)
     best_p = argmin_p r_p
     eta[best_p] += nu * cand_pred[best_p]
     selection_order.push(best_p)
     risk[step+1] = family.risk(y, etas, w)
```

The per-parameter selection counts sum to `mstop`, so a single scalar tunes all
parameters. The trial-risk loop clones `eta[p]` temporarily (in place) and
restores it — a small $O(K)$ per-iteration penalty over the cyclic driver.

## Sequential offset computation

Offsets are computed **in parameter order** with a shared `prior` vector. At
`p = k`, `prior[0..k-1]` holds the offsets already computed; later elements are
uninitialized. The family uses defaults for later parameters (usually the link
of the mean, median, or a zero-inflated proportion).

```rust
// lss.rs:153-159 — sequential offset with prior threading
let mut offs: Vec<f64> = Vec::with_capacity(k);
for p in 0..k {
    let off = family.offset(p, ys, ws, &offs);  // offs = prior
    offs.push(off);
    etas.push(Array1::from_elem(n, off));
}
```


Examples of sequential dependence:

| Family | $\mu$ offset | $\sigma$ offset uses $\mu$ offset |
|---|---|---|
| `GaussianLSS` | weighted mean $\bar y_w$ | $\sqrt{\sum w_i (y_i - \bar y_w)^2 / \sum w_i}$ (residual std) |
| `StudentTLSS` | weighted median (robust — not the mean) | robust scale via MAD of residuals from $F^{-1}_{t_3}(0.5)$ |
| `GammaLSS` | $\log(\bar y_w)$ | $\log(\text{shape})$ via gamma MLE at fixed $\mu$ |
| `ZIPoLSS` | $\log(\bar y)$ | logit of zero fraction $(n_0 + 1)/(n + 2)$ |
| `DirichletLSS` | $\min_i w_i y_{i,k}$ (per-column) | independent ($\mu$-like offset for each $a_k$) |

The `StudentTLSS` $\mu$ offset is a **robust location** (weighted median), not
the sample mean. R's `gamboostLSS` computes it as the minimizer of
$\sum w_i \rho(y_i - c)$ where $\rho$ is the $t$-loss with fixed $\sigma=1$ and
$\text{df}=3$. A naive arithmetic-mean offset gives visibly different paths.

## Multi-column response (DirichletLSS)

Families with $K>1$ naturally map to a $K$-column response matrix. The engine
accepts a **concatenated slice** `y = [col_0, col_1, ..., col_{K-1}]` of length
$K \cdot n$. The observation count is derived from the **weights** (always length
$n$), not from `y.len()`:

```rust
// lss.rs:126 — observation count from weights
let n = ws.len();
```


This lets `DirichletLSS` address column $k$'s $i$-th observation at index
`k * n + i`:

```rust
// lss.rs:1729-1736 — DirichletLSS ngradient with column slicing
fn ngradient(&self, k: usize, y: &[f64], etas: &[Array1<f64>], _w: &[f64], out: &mut [f64]) {
    let n = self.n_obs(y);  // y.len() / self.k
    for i in 0..n {
        let alpha_k = etas[k][i].exp();
        let sum_alpha = (0..self.k).map(|j| etas[j][i].exp()).sum();
        out[i] = alpha_k * (digamma(sum_alpha) - digamma(alpha_k) + y[k * n + i].ln());
    }
}
```


The same layout applies to survival AFT families (`LogNormalLSS`, `LogLogLSS`,
`WeibullLSS`), which carry a 2-column `(time, status)` response with $K=2$
parameters $(\mu, \sigma)$.

## Cross-validated risk over the grid

`cvrisk_lss` (`lss.rs:475-509`) evaluates the joint out-of-bag negative
log-likelihood over a grid of per-parameter `mstop` vectors.

A grid point with unequal parameter budgets generally requires its own fit. If
parameter $k$ stops early, subsequent gradients for the remaining parameters
change; summing prefixes from a longer all-parameter path would score a different
model. Diagonal points $(m,\ldots,m)$ are the exception: they are checkpoints of
one cyclic path and are collected in a single run up to the largest requested
$m$.

For every fold, `cvrisk_lss_with` calls the learner blueprints with the in-bag
weights. This recomputes centering, `df_to_lambda`, and the factorization while
retaining the cached spline basis and penalty. Keeping the full-sample lambda or
centering would produce a small but systematic disagreement with R. Fold
preparation is fallible: a degenerate fold returns a frontend error instead of
panicking inside a parallel worker.

```rust
// lss.rs:539-550 — parallel fold evaluation with per-fold learner rebuild
let rows: Vec<Vec<f64>> = (0..b).into_par_iter().map(|fold| {
    let fold_w = extract_fold_weights(folds, fold, n);
    let learners = prepare(fold_w)?;  // fold-specific lambda and factors
    let diagonal = fit_diagonal_checkpoints(learners, diagonal_mstop);
    grid.iter().map(|g| {
        let eta = if all_equal(g) { diagonal[g[0]] }
                  else { fit_lss(learners, y, fold_w, family, g)?.fitted };
        Ok(family.risk(y, &eta, &oob_w))
    }).collect()
}).collect();
```


The predictor $\eta_k^{(m)} = \text{offset}_k + \nu \sum_{t=1}^{m_k} X^{(t)}_{j^\star_{k,t}} \beta^{(t)}_{k,t}$ is reconstructed from coefficients of the path fitted to that
budget vector. Diagonal checkpoints reuse one path; unequal budgets use their
separate fits.

## Adding an LSS family

A new family is a single `impl LssFamily` with these methods (`lss.rs:57-89`):

| Method | Purpose |
|---|---|
| `n_params()` | Number of distribution parameters $K$ |
| `param_names()` | Labels for each parameter (e.g. `["mu","sigma"]`) |
| `offset(k, y, w, prior)` | Sequential offset for parameter $k$ |
| `ngradient(k, y, etas, w, out)` | Partial negative gradient w.r.t. $\eta_k$ |
| `risk(y, etas, w)` | Joint negative log-likelihood |
| `response(k, eta, out)` | Inverse link: $\theta_k = g_k^{-1}(\eta_k)$ |
| `check_y(y)` | Validate response domain (e.g. $y>0$, $0<y<1$) |

The drivers, `cvrisk`, and stability selection then work unchanged. The
families implemented this way include the zero-inflated count models
(`ZIPoLSS`, `ZINBLSS`) and the compositional `DirichletLSS`, whose response
is carried as a concatenated $K$-column slice.

A test helper `assert_ngradient_matches_finite_diff` (`lss.rs:3134-3155`)
verifies each analytic gradient against central finite differences:

```rust
// lss.rs:3134-3155 — finite-difference gradient verification
fn assert_ngradient_matches_finite_diff(fam: &dyn LssFamily, y, etas, n) {
    let eps = 1e-6;
    for k in 0..fam.n_params() {
        fam.ngradient(k, y, etas, &w, &mut ngr);
        for j in 0..n {
            let saved = etas[k][j];
            etas[k][j] = saved + eps;  r_plus = fam.risk(y, etas, w);
            etas[k][j] = saved - eps;  r_minus = fam.risk(y, etas, w);
            let fd = (r_minus - r_plus) / (2.0 * eps); // -d(risk)/d(eta)
            assert!((ngr[j] - fd).abs() < 1e-5, "...");
            etas[k][j] = saved;
        }
    }
}
```


This catches $\partial \ell / \partial \eta_k$ chain-rule errors, especially in
families with intricate gradients like `ZINBLSS` (3 parameters, logit link for
$\nu$, where R's gradient differs from the analytical derivation).

## See also

- [](architecture.ipynb) — how the LSS trait plugs into the overall crate architecture.
- [](../methodology/gamlss) — the GAMLSS framework and parametrizations.
- [](../methodology/gamlss_framework) — the joint modeling principles behind the LSS engine.